# Joint BRDF + drift fit — end-to-end smoke test

This notebook applies the **four-parameter joint BRDF+drift regression** (the
whiteboard system in `brdf_exploration.ipynb`) to **Libya_4 patch `10_7_1`** and
walks through interpreting the output.

> **This is a smoke test on ~13 QC-passed scenes.** The goal is to verify the
> machinery runs end to end and returns sensible, correctly-signed numbers — **not**
> to estimate drift. Every uncertainty here is plumbing verification: 13 scenes is
> at the validity floor for the bootstrap, and the drift/geometry regressors are
> strongly collinear at this sample size. Treat every interval as "does the code
> produce a finite, plausible range", never as a scientific result.

It reuses the repo's conventions from `simple_linear_BRDF.ipynb`: the geometry
table + QC gate for scene selection, `nearest_channel` for the per-scene
patch-median value at one wavelength, one row per scene, and matplotlib / numpy /
pandas / scipy only.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------------------------------------------
# USER-CONFIGURABLE INPUTS
# ------------------------------------------------------------------
PICS_ID       = "Libya_4"     # underscored site id
PATCH_ID      = "10_7_1"      # patch_id within the PICS
WAVELENGTH_NM = 1050.0        # target wavelength (nm); nearest channel is used
Y_STAT        = "median"      # patch statistic for the y-value

N_BOOT        = 1000          # scene-level bootstrap resamples
RNG_SEED      = 42            # fixed throughout for reproducibility
T0            = "2022-08-01"  # global epoch; t is decimal years since T0
REF_SZA_DEG   = 30.0          # reference geometry for the drift-rate combination

SZA_STAT = "sza_mean"         # per-scene SZA column in the geometry table
VZA_STAT = "vza_mean"         # per-scene VZA column (used only in diagnostics)
APPLY_QC = True               # drop keep=FALSE scenes (falls back to all if no sheet)

# ------------------------------------------------------------------
# Repo-relative paths (walk up until we find the package dir).
# ------------------------------------------------------------------
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "emit_pics").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
GEOM_PATH = DATA_ROOT / "geometry" / f"geometry_{PICS_ID}_{PATCH_ID}.csv"
RFL_DIR   = DATA_ROOT / "reflectance"
RAD_DIR   = DATA_ROOT / "radiance"
QC_PATH   = DATA_ROOT / "qc" / f"{PICS_ID}_{PATCH_ID}_qc.csv"

# Section-by-section verdicts, printed and aggregated at the end.
VERDICTS = {}

plt.rcParams["figure.dpi"] = 110
print(f"Patch {PICS_ID}/{PATCH_ID}  @ {WAVELENGTH_NM:.0f} nm ({Y_STAT})  epoch t0={T0}")
print(f"Geometry table exists: {GEOM_PATH.exists()}")

## 2. The model (stated before any fitting)

The whiteboard system factors an observation into an instrument response that
**drifts in time** and a surface response that **depends on geometry**:

$$R_{t} = a + b\,t \qquad\text{(Line 1: instrument, drifts with } t\text{)}$$
$$\rho_{t,\theta} = (c + d\,g)\,R_{t} \qquad\text{(Line 2: surface bracket} \times \text{instrument)}$$

where $g$ is the **geometry regressor** — either $\cos\theta$ or the raw SZA in
degrees. Expanding Line 2 gives the linear model we actually fit:

$$\boxed{\;\rho = \alpha + \beta\,t + \gamma\,g + \eta\,(g\,t)\;}$$

with the four coefficients mapping to the physical parameters as

$$\alpha = a c,\quad \beta = b c,\quad \gamma = a d,\quad \eta = b d .$$

**Derived quantities we report (not the raw coefficients):**

- **Fractional drift at a fixed reference geometry** $g_{\text{ref}}$:
  $$\frac{\partial_t \rho}{\rho}\Big|_{g_{\text{ref}}}
     = \frac{\beta + \eta\,g_{\text{ref}}}{\alpha + \gamma\,g_{\text{ref}}}\;\;(\%/\text{yr}).$$
  Reported at a **fixed** $g_{\text{ref}}$ (SZA = 30°), *not* the sample mean, so the
  four runs are comparable.
- **Fractional directional sensitivity** $\gamma/\alpha = d/c$.
- **Multiplicative-structure consistency**: since $\alpha\eta = acbd = \beta\gamma$,
  the ratio $\alpha\eta/(\beta\gamma)$ should be $\approx 1$ **if** the separable
  structure holds. Expect it to be extremely noisy at $n=13$.

> The analytical standard errors below come from the OLS covariance
> $\sigma^2 (X^\top X)^{-1}$. They are **provisional** — with $n=13$ and collinear
> $t$ vs $g t$ they understate reality. The **scene-level bootstrap** (§5) is the
> uncertainty to read.

**Four runs**, every combination of $y \in \{\text{reflectance}, \text{radiance}\}$
and $g \in \{\cos(\text{SZA}), \text{SZA}^\circ\}$, driven by a single
`fit_joint(df, ycol, gcol)`.

## 3. Load scenes, apply the QC gate, build one row per scene

Same pattern as `simple_linear_BRDF.ipynb`: scenes from the geometry table,
`apply_qc_filter(require_review=False)`, and `nearest_channel` for the per-scene
patch-median reflectance and radiance at the target wavelength. Radiance-missing
scenes keep a NaN there and are dropped only from the radiance runs.

In [ ]:
from emit_pics.qc_gate import apply_qc_filter

def nearest_channel(csv_path, target_nm, stat_col):
    # (actual_wl, value) at the channel nearest target_nm, or (nan, nan).
    if not csv_path.exists():
        return np.nan, np.nan
    d = pd.read_csv(csv_path)
    if stat_col not in d.columns or "central_wl" not in d.columns:
        return np.nan, np.nan
    i = (d["central_wl"] - target_nm).abs().idxmin()
    return float(d["central_wl"].iloc[i]), float(d[stat_col].iloc[i])

if not GEOM_PATH.exists():
    raise FileNotFoundError(
        f"Geometry table not found:\n  {GEOM_PATH}\n"
        "Run the extraction step for this patch first (CACHE_PIXELS not required — "
        "the per-channel + geometry CSVs suffice)."
    )

scenes = pd.read_csv(GEOM_PATH).sort_values("datetime").reset_index(drop=True)
if APPLY_QC:
    scenes, qc_rep = apply_qc_filter(scenes, QC_PATH, require_review=False,
                                     timestamp_col="timestamp")
    print(f"QC: {qc_rep['n_before']} -> {qc_rep['n_after']} scenes "
          f"({qc_rep['n_excluded']} excluded {qc_rep['reasons']})")
    if not QC_PATH.exists():
        print(f"  (no QC worksheet; using all scenes)")

t0 = pd.to_datetime(T0, utc=True)
has_vza = VZA_STAT in scenes.columns

rows = []
for _, s in scenes.iterrows():
    ts = str(s["timestamp"])
    _, rfl = nearest_channel(RFL_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rfl.csv",
                             WAVELENGTH_NM, f"{Y_STAT}_rfl")
    _, rad = nearest_channel(RAD_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rad.csv",
                             WAVELENGTH_NM, f"{Y_STAT}_rad")
    dt = pd.to_datetime(s["datetime"], utc=True, errors="coerce")
    rows.append({
        "timestamp": ts,
        "t": (dt - t0).total_seconds() / (365.25 * 86400.0),
        "sza": float(s[SZA_STAT]),
        "vza": float(s[VZA_STAT]) if has_vza else np.nan,
        "rfl": rfl, "rad": rad,
    })

df = pd.DataFrame(rows)
df["cos_sza"] = np.cos(np.radians(df["sza"]))

n_rfl = int(df["rfl"].notna().sum())
n_rad = int(df["rad"].notna().sum())
print(f"{len(df)} scene(s): {n_rfl} with reflectance, {n_rad} with radiance; "
      f"VZA column {'present' if has_vza else 'ABSENT'}")
print(f"t span {df['t'].min():.2f}..{df['t'].max():.2f} yr, "
      f"SZA {df['sza'].min():.1f}..{df['sza'].max():.1f} deg")

_v = "PASS" if n_rfl >= 8 else "CHECK"
VERDICTS["3-data"] = _v
print(f"[3] {_v}: {n_rfl} reflectance scenes loaded "
      f"({'enough to exercise the 4-param fit' if _v == 'PASS' else 'thin — fit may be ill-posed'})")
df.head()

## 4. The joint fit, run over all four combinations

One pure `fit_joint(df, ycol, gcol)` builds the design matrix
`[1, t, g, g*t]`, solves by least squares, and returns coefficients, covariance,
residuals, and the derived quantities. It is looped over the four `(y, g)` combos
— no copy-pasted blocks. Runs with too few finite rows (e.g. radiance absent)
degrade to a flagged, NaN-filled result instead of raising.

In [ ]:
def _solve(X, y):
    # OLS via least squares; covariance from sigma^2 (X'X)^+ (pinv = robust to
    # near-singular design at small n). Returns (coef, cov, resid).
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ coef
    dof = len(y) - X.shape[1]
    XtX_inv = np.linalg.pinv(X.T @ X)
    sigma2 = float(resid @ resid) / dof if dof > 0 else np.nan
    return coef, sigma2 * XtX_inv, resid

def g_ref_for(gcol):
    # Fixed reference geometry per regressor, plus a human label.
    if gcol == "cos_sza":
        gr = float(np.cos(np.radians(REF_SZA_DEG)))
        return gr, f"cos({REF_SZA_DEG:.0f} deg)={gr:.3f}"
    return float(REF_SZA_DEG), f"SZA={REF_SZA_DEG:.0f} deg"

def _derived(coef, g_ref):
    a, b, gm, et = coef
    denom = a + gm * g_ref
    frac = (b + et * g_ref) / denom if denom != 0 else np.nan
    dcs = gm / a if a != 0 else np.nan
    cons = (a * et) / (b * gm) if (b * gm) != 0 else np.nan
    return frac, dcs, cons

def fit_joint(df, ycol, gcol):
    sub = df.dropna(subset=["t", gcol, ycol]).reset_index(drop=True)
    n_drop = len(df) - len(sub)
    g_ref, g_ref_label = g_ref_for(gcol)
    out = {"ycol": ycol, "gcol": gcol, "n": len(sub), "n_dropped": n_drop,
           "g_ref": g_ref, "g_ref_label": g_ref_label, "sub": sub}
    if len(sub) < 5:                      # need dof >= 1 for 4 parameters
        out.update(insufficient=True, coef=np.full(4, np.nan), se=np.full(4, np.nan),
                   resid=np.array([]), alpha=np.nan, beta=np.nan, gamma=np.nan,
                   eta=np.nan, frac_drift=np.nan, dir_sens=np.nan,
                   consistency=np.nan, rmse=np.nan, mean_y=np.nan)
        return out
    t = sub["t"].to_numpy(float); g = sub[gcol].to_numpy(float); y = sub[ycol].to_numpy(float)
    X = np.column_stack([np.ones_like(t), t, g, g * t])
    coef, cov, resid = _solve(X, y)
    frac, dcs, cons = _derived(coef, g_ref)
    out.update(insufficient=False, coef=coef, se=np.sqrt(np.diag(cov)), cov=cov,
               resid=resid, alpha=coef[0], beta=coef[1], gamma=coef[2], eta=coef[3],
               frac_drift=frac, dir_sens=dcs, consistency=cons,
               rmse=float(np.sqrt(np.mean(resid ** 2))), mean_y=float(np.mean(y)))
    return out

def run_label(ycol, gcol):
    return f"{ycol}/{'cos' if gcol == 'cos_sza' else 'SZA'}"

RUNS = [("rfl", "cos_sza"), ("rfl", "sza"), ("rad", "cos_sza"), ("rad", "sza")]
results = {rg: fit_joint(df, *rg) for rg in RUNS}

for rg in RUNS:
    r = results[rg]
    lab = run_label(*rg)
    if r["insufficient"]:
        print(f"  {lab:12s}: INSUFFICIENT (n={r['n']}, dropped {r['n_dropped']}) - skipped")
        continue
    a, b, gm, et = r["coef"]; sa, sb, sg, se_ = r["se"]
    print(f"  {lab:12s}: n={r['n']:2d} drop={r['n_dropped']:2d} | "
          f"a={a:+.4g}({sa:.2g}) b={b:+.3g}({sb:.2g}) "
          f"g={gm:+.4g}({sg:.2g}) e={et:+.3g}({se_:.2g})")

_ok = all(not results[rg]["insufficient"] for rg in RUNS if rg[0] == "rfl")
_v = "PASS" if _ok else "FAIL"
VERDICTS["4-fit"] = _v
print(f"[4] {_v}: reflectance runs produced finite coefficients "
      f"({'radiance runs may degrade if CSVs absent' if _ok else 'reflectance fit failed'})")

## 5a. Per-run coefficients and derived quantities

For each run: the four Greek coefficients (with provisional SEs), the fractional
drift at the fixed reference geometry, the directional sensitivity $\gamma/\alpha$,
and the consistency ratio $\alpha\eta/\beta\gamma$. The reference geometry used is
labeled per run (cos runs use $\cos30^\circ$; degree runs use $30^\circ$).

In [ ]:
def per_run_row(rg):
    r = results[rg]
    row = {"run": run_label(*rg), "n": r["n"], "g_ref": r["g_ref_label"]}
    if r["insufficient"]:
        row.update(alpha=np.nan, beta=np.nan, gamma=np.nan, eta=np.nan,
                   drift_pct_yr=np.nan, dir_sens=np.nan, consistency=np.nan)
        return row
    row.update(alpha=r["alpha"], beta=r["beta"], gamma=r["gamma"], eta=r["eta"],
               drift_pct_yr=100 * r["frac_drift"], dir_sens=r["dir_sens"],
               consistency=r["consistency"])
    return row

report = pd.DataFrame([per_run_row(rg) for rg in RUNS])
with pd.option_context("display.float_format", lambda v: f"{v:.4g}"):
    print(report.to_string(index=False))

# Sign expectations: cos runs -> gamma>0 (brighter at high sun / large cos);
# degree runs -> gamma<0 (brighter at low SZA). Check on the runs that fit.
def _sign_ok(rg):
    r = results[rg]
    if r["insufficient"]:
        return True                       # can't contradict; don't fail on it
    return (r["gamma"] > 0) if rg[1] == "cos_sza" else (r["gamma"] < 0)

bad = [run_label(*rg) for rg in RUNS if not _sign_ok(rg)]
_v = "PASS" if not bad else "CHECK"
VERDICTS["5-signs"] = _v
print(f"\n[5a] {_v}: geometry-coefficient signs "
      f"{'all as expected (gamma>0 for cos, gamma<0 for SZA)' if not bad else 'unexpected in: ' + ', '.join(bad)}")

## 5b. Scene-level bootstrap

Resample the scene rows with replacement, refit, `N_BOOT` times; report percentile
95% CIs for the fractional drift and $\gamma/\alpha$. Degenerate resamples (fewer
than 4 distinct scenes, or zero spread in $t$ or $g$) cannot support the
four-parameter fit — they are skipped and counted, and the skip fraction is
printed.

> **Warning.** 13 scenes is at the validity floor for a bootstrap. These intervals
> verify that the resampling plumbing runs and yields finite, ordered bounds — they
> are **not** trustworthy confidence statements about drift.

In [ ]:
def bootstrap_run(res, n_boot, rng):
    if res["insufficient"]:
        return np.array([]), np.array([]), n_boot
    sub = res["sub"]; g_ref = res["g_ref"]
    y = sub[res["ycol"]].to_numpy(float)
    g = sub[res["gcol"]].to_numpy(float)
    t = sub["t"].to_numpy(float)
    idx = np.arange(len(sub))
    fr, ds, skipped = [], [], 0
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        if len(np.unique(s)) < 4 or np.ptp(g[s]) == 0 or np.ptp(t[s]) == 0:
            skipped += 1
            continue
        X = np.column_stack([np.ones_like(t[s]), t[s], g[s], g[s] * t[s]])
        coef, _, _ = _solve(X, y[s])
        f, d, _c = _derived(coef, g_ref)
        if not np.isfinite(f):
            skipped += 1
            continue
        fr.append(f); ds.append(d)
    return np.array(fr, float), np.array(ds, float), skipped

rng = np.random.default_rng(RNG_SEED)
BOOT = {}
boot_rows = []
for rg in RUNS:
    fr, ds, sk = bootstrap_run(results[rg], N_BOOT, rng)
    BOOT[rg] = {"frac": fr, "dir": ds, "skipped": sk}
    if fr.size:
        f_lo, f_hi = np.nanpercentile(fr * 100, [2.5, 97.5])
        d_lo, d_hi = np.nanpercentile(ds, [2.5, 97.5])
    else:
        f_lo = f_hi = d_lo = d_hi = np.nan
    boot_rows.append({"run": run_label(*rg), "n_boot_ok": fr.size,
                      "skip_frac": sk / N_BOOT,
                      "drift_CI_lo%/yr": f_lo, "drift_CI_hi%/yr": f_hi,
                      "dc_CI_lo": d_lo, "dc_CI_hi": d_hi})

boot_tbl = pd.DataFrame(boot_rows)
with pd.option_context("display.float_format", lambda v: f"{v:.4g}"):
    print(boot_tbl.to_string(index=False))

_maxskip = max(BOOT[rg]["skipped"] / N_BOOT for rg in RUNS if not results[rg]["insufficient"])
_v = "PASS" if _maxskip < 0.20 else "CHECK"
VERDICTS["5-boot"] = _v
print(f"\n[5b] {_v}: max skip fraction {_maxskip:.1%} "
      f"({'resampling well-posed' if _v == 'PASS' else 'many degenerate resamples — sample too thin'})")

## 6. Diagnostics

Reference run for the plots is **reflectance / cos(SZA)**. Structureless residuals
and low time-geometry correlation are what "pass" looks like.

### 6a. rho vs t, colored by SZA, with the drift line at the reference geometry

In [ ]:
res = results[("rfl", "cos_sza")]
if not res["insufficient"]:
    sub = res["sub"]
    fig, ax = plt.subplots(figsize=(7, 4.4))
    sc = ax.scatter(sub["t"], sub["rfl"], c=sub["sza"], cmap="viridis",
                    s=45, edgecolor="k", linewidth=0.3)
    tg = np.linspace(sub["t"].min(), sub["t"].max(), 100)
    line = (res["alpha"] + res["gamma"] * res["g_ref"]) + \
           (res["beta"] + res["eta"] * res["g_ref"]) * tg
    ax.plot(tg, line, "r--", lw=1.6, label=f"drift @ {res['g_ref_label']}")
    plt.colorbar(sc, label="SZA (deg)")
    ax.set_xlabel(f"t (years since {T0})")
    ax.set_ylabel(f"reflectance ({WAVELENGTH_NM:.0f} nm, {Y_STAT})")
    ax.set_title("rho vs t  (reflectance / cos run)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()
    print(f"[6a] PASS: drift line at {res['g_ref_label']} overlaid; "
          f"drift {100 * res['frac_drift']:+.3f}%/yr (inspect scatter vs line)")
else:
    print("[6a] CHECK: reflectance/cos run insufficient; nothing to plot")

### 6b. Residuals vs t, vs g, and vs VZA (structureless = pass)

In [ ]:
res = results[("rfl", "cos_sza")]
if not res["insufficient"]:
    sub = res["sub"]; resid = res["resid"]
    has_v = sub["vza"].notna().any()
    panels = [("t", sub["t"].to_numpy(float), "t (yr)"),
              ("g", sub["cos_sza"].to_numpy(float), "cos(SZA)")]
    if has_v:
        panels.append(("vza", sub["vza"].to_numpy(float), "VZA (deg)"))
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.8), sharey=True)
    axes = np.atleast_1d(axes)
    max_abs_r = 0.0
    for ax, (_key, xv, xlab) in zip(axes, panels):
        ax.scatter(xv, resid, s=28, edgecolor="k", linewidth=0.3)
        ax.axhline(0.0, color="0.6", lw=0.8, ls=":"); ax.set_xlabel(xlab); ax.grid(alpha=0.25)
        m = np.isfinite(xv) & np.isfinite(resid)
        if m.sum() >= 3 and np.ptp(xv[m]) > 0:
            max_abs_r = max(max_abs_r, abs(stats.pearsonr(xv[m], resid[m])[0]))
    axes[0].set_ylabel("reflectance residual")
    fig.suptitle("6b  Joint-fit residuals"); plt.tight_layout(); plt.show()
    _v = "PASS" if max_abs_r < 0.5 else "CHECK"
    VERDICTS["6-resid"] = _v
    print(f"[6b] {_v}: max |corr(residual, regressor)| = {max_abs_r:.3f} "
          f"({'structureless' if _v == 'PASS' else 'residual structure remains'})"
          f"{'' if has_v else '  (no VZA column; 2 panels)'}")
else:
    VERDICTS["6-resid"] = "CHECK"
    print("[6b] CHECK: reflectance/cos run insufficient")

### 6c. Conditioning: corr(t, SZA) and corr(t, VZA)

In [ ]:
def _corr(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3 or np.ptp(a[m]) == 0 or np.ptp(b[m]) == 0:
        return np.nan
    return stats.pearsonr(a[m], b[m])[0]

r_tsza = _corr(df["t"], df["sza"])
r_tvza = _corr(df["t"], df["vza"])
print(f"corr(t, SZA) = {r_tsza:+.3f}")
print(f"corr(t, VZA) = {r_tvza:+.3f}" if np.isfinite(r_tvza) else "corr(t, VZA) = n/a (no VZA)")

# Threshold from the simple_linear_BRDF diagnostics: |r| < 0.3 well conditioned.
_v = "PASS" if (np.isfinite(r_tsza) and abs(r_tsza) < 0.3) else "CHECK"
VERDICTS["6-cond"] = _v
print(f"[6c] {_v}: |corr(t, SZA)| = {abs(r_tsza):.3f} "
      f"({'well conditioned; drift and geometry separable' if _v == 'PASS' else 'entangled — drift/geometry hard to separate, widen intervals'})")

### 6d. Bootstrap distribution of the fractional drift (reflectance / cos)

In [ ]:
b = BOOT[("rfl", "cos_sza")]
res = results[("rfl", "cos_sza")]
if b["frac"].size and not res["insufficient"]:
    fr = b["frac"] * 100.0
    lo, hi = np.nanpercentile(fr, [2.5, 97.5])
    point = 100.0 * res["frac_drift"]
    fig, ax = plt.subplots(figsize=(7, 4.0))
    ax.hist(fr, bins=40, color="#9ecae1", edgecolor="0.4")
    ax.axvline(point, color="#d62728", lw=1.8, label=f"point {point:+.3f}%/yr")
    ax.axvline(lo, color="k", ls="--", lw=1.2, label=f"95% CI [{lo:+.2f}, {hi:+.2f}]")
    ax.axvline(hi, color="k", ls="--", lw=1.2)
    ax.set_xlabel("fractional drift (%/yr)"); ax.set_ylabel("bootstrap count")
    ax.set_title("6d  Bootstrap drift (reflectance / cos)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25); plt.tight_layout(); plt.show()
    _v = "PASS" if np.isfinite(lo) and np.isfinite(hi) and lo <= point <= hi else "CHECK"
    print(f"[6d] {_v}: CI [{lo:+.2f}, {hi:+.2f}] %/yr brackets point {point:+.3f} "
          f"({'finite and ordered' if _v == 'PASS' else 'point outside CI — heavy skew, expected at n=13'})")
else:
    print("[6d] CHECK: no usable bootstrap draws for reflectance/cos")

## 7. Comparison across the four runs

**Read this table as plumbing, not physics.**

- **rad vs rfl** answer *different physical questions*: radiance drift is the
  instrument-only response; reflectance drift folds in ISOFIT / atmospheric
  retrieval, so they need not match.
- **SZA vs cos(SZA)** runs are expected to be **nearly identical** at $n=13$: the
  two regressors correlate ~0.99 over this SZA range, so any difference between
  them here is **noise, not evidence** for either regressor. (Signs differ because
  $\cos$ increases while SZA decreases toward zenith; the *magnitudes* of the
  physics — drift, $|d/c|$ — are what should agree.)

In [ ]:
def cmp_row(rg):
    r = results[rg]; bt = BOOT[rg]
    row = {"run": run_label(*rg), "n": r["n"]}
    if r["insufficient"]:
        row.update(drift_pct_yr=np.nan, drift_CI="n/a", dir_sens=np.nan,
                   consistency=np.nan, rmse_pct=np.nan)
        return row
    if bt["frac"].size:
        lo, hi = np.nanpercentile(bt["frac"] * 100, [2.5, 97.5])
        ci = f"[{lo:+.2f}, {hi:+.2f}]"
    else:
        ci = "n/a"
    row.update(drift_pct_yr=100 * r["frac_drift"], drift_CI=ci,
               dir_sens=r["dir_sens"], consistency=r["consistency"],
               rmse_pct=100 * r["rmse"] / r["mean_y"] if r["mean_y"] else np.nan)
    return row

cmp_tbl = pd.DataFrame([cmp_row(rg) for rg in RUNS])
with pd.option_context("display.float_format", lambda v: f"{v:.4g}"):
    print(cmp_tbl.to_string(index=False))

# Consistency should be order-of-magnitude ~1 if separable; noisy at n=13.
_cons = [abs(results[rg]["consistency"]) for rg in RUNS
         if not results[rg]["insufficient"] and np.isfinite(results[rg]["consistency"])]
_oom = [c for c in _cons if 0.1 <= c <= 10.0]
_v = "PASS" if _cons and len(_oom) >= 1 else "CHECK"
VERDICTS["7-compare"] = _v
print(f"\n[7] {_v}: {len(_oom)}/{len(_cons)} runs have |alpha*eta/beta*gamma| within [0.1, 10] "
      f"(order-of-magnitude ~1 as separability predicts; heavy noise expected at n=13)")

### 7b. Cross-check $\gamma/\alpha$ against the simple marginal fit

The 2-parameter marginal fit $\rho = A + B g$ (the `simple_linear_BRDF`
convention) gives $B/A$, which should roughly equal the joint fit's
$\gamma/\alpha$ **when `corr(t, SZA)` is low** — because then the time terms don't
lean on the geometry terms. We refit the marginal inline on the *same* rows (no
import from the other notebook).

In [ ]:
res = results[("rfl", "cos_sza")]
if not res["insufficient"]:
    sub = res["sub"]
    B, A = np.polyfit(sub["cos_sza"].to_numpy(float), sub["rfl"].to_numpy(float), 1)
    marg = B / A if A != 0 else np.nan          # slope / intercept
    joint = res["dir_sens"]                       # gamma / alpha
    pct = 100 * (joint - marg) / marg if marg not in (0, np.nan) else np.nan
    print(f"marginal (rho vs cos)  B/A       = {marg:+.4g}")
    print(f"joint fit              gamma/alpha = {joint:+.4g}")
    print(f"difference                        = {pct:+.1f}%")
    _v = "PASS" if np.isfinite(pct) and abs(pct) < 20 else "CHECK"
    VERDICTS["7-xcheck"] = _v
    print(f"[7b] {_v}: marginal vs joint d/c differ by {abs(pct):.1f}% "
          f"({'agree as expected under low t-SZA correlation' if _v == 'PASS' else 'divergence — check conditioning (6c)'})")
else:
    VERDICTS["7-xcheck"] = "CHECK"
    print("[7b] CHECK: reflectance/cos run insufficient")

## 8. Interpretation (for a reader new to the framework)

**The four Greek coefficients** — each is one "bin" of the expansion
$\rho = \alpha + \beta t + \gamma g + \eta (g t)$:

- $\alpha$ — **baseline**: the signal at $t=0$ and $g=0$ (the $a c$ product).
- $\beta$ — **time**: how the signal drifts per year, *before* geometry weighting
  ($b c$).
- $\gamma$ — **geometry**: how the signal changes with the geometry regressor ($a d$).
- $\eta$ — **geometry–time echo**: the interaction ($b d$) — how the *drift itself*
  bends with geometry. It is the smallest, noisiest term and the whole reason the
  raw coefficients are hard to read individually.

**Why $\beta/\alpha$ raw is noisy.** The columns $t$ and $g t$ are strongly
collinear (they differ only by the slowly-varying $g$), so the fit cannot cleanly
attribute variance between $\beta$ and $\eta$. Their individual SEs balloon, and
$\beta/\alpha$ alone inherits that instability.

**Why we report drift at a reference geometry instead.** The combination
$(\beta + \eta g_{\text{ref}})/(\alpha + \gamma g_{\text{ref}})$ is the quantity
that is actually well-determined: it is the model's predicted fractional drift at a
*fixed, physically meaningful* geometry (SZA = 30°). Evaluating all runs at the
same $g_{\text{ref}}$ also makes them comparable, which the sample-mean geometry
would not.

**What PASS looks like for this smoke test** (machinery works, numbers sane):

- $\gamma > 0$ for the $\cos$ runs, $\gamma < 0$ for the degree runs (sign flips
  with the regressor's direction).
- The drift CI is **wide but finite and ordered** — plumbing works even though the
  number is not trustworthy.
- Consistency ratio $\alpha\eta/\beta\gamma$ is **order-of-magnitude ~1** (not
  exactly 1 — expect large scatter at $n=13$).
- Residuals are **structureless** vs $t$, $g$, and VZA.
- `corr(t, SZA)` is low ($|r|<0.3$), and marginal vs joint $d/c$ roughly agree.

**What this run CANNOT conclude** — do not carry any of these forward:

- **No drift claim.** 13 scenes, collinear regressors: the drift number and its CI
  are verification artifacts, not a measurement.
- **No SZA-vs-cos(SZA) verdict.** The regressors correlate ~0.99 here; the two
  runs are interchangeable and cannot adjudicate the better geometry model.
- **No aggregation-level decision.** One patch, one wavelength, patch-median
  y-values — this says nothing about pixel- vs patch-level fitting, wavelength
  choice, or cross-patch pooling.

*(Markdown section — no verdict line; the aggregate verdict is below.)*

## 9. Aggregate smoke-test verdict

In [ ]:
order = ["3-data", "4-fit", "5-signs", "5-boot", "6-resid", "6-cond",
         "7-compare", "7-xcheck"]
print("Section verdicts:")
for k in order:
    print(f"  {k:12s}: {VERDICTS.get(k, '-')}")

_vals = [VERDICTS.get(k, "-") for k in order]
if "FAIL" in _vals:
    overall = "FAIL"
elif _vals.count("CHECK") >= 3:
    overall = "CHECK"
else:
    overall = "PASS"
n_check = _vals.count("CHECK")
print(f"\n[SMOKE TEST] {overall}: {_vals.count('PASS')} PASS / {n_check} CHECK / "
      f"{_vals.count('FAIL')} FAIL across {len(order)} sections "
      f"— {'machinery verified end to end (numbers are NOT a drift result)' if overall != 'FAIL' else 'fit machinery did not complete; investigate above'}")